<a href="https://colab.research.google.com/github/Udon-datascience/PyTorch_study_log/blob/main/code/6_DataLoader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#DataLoaderにより手書き文字の認識を行う

In [ ]:
from torchvision.datasets import MNIST
from torchvision import transforms

img_size = 28     #画像の高さと幅を指定

#訓練データを取得し、Tensorに変換
mnist_train = MNIST("./data", train=True, download=True, transform=transforms.ToTensor())

#テストデータを取得し、Tensorに変換
mnist_test = MNIST("./data", train=False, download=True, transform=transforms.ToTensor())

print("訓練データの数：", len(mnist_train), "テストデータの数：", len(mnist_test))

100%|██████████| 9.91M/9.91M [00:00<00:00, 12.5MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 337kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.14MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 12.6MB/s]

訓練データの数： 60000 テストデータの数： 10000


In [ ]:
from torch.utils.data import DataLoader

#DataLoaderの設定
batch_size = 256    #バッチサイズ
train_loader = DataLoader(mnist_train, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(mnist_test, batch_size=batch_size, shuffle=False)

In [ ]:
import torch.nn as nn

class Net(nn.Module):
  def __init__(self):
    super().__init__()
    self.fcl = nn.Linear(img_size*img_size, 1024)
#全結合層
    self.fc2 = nn.Linear(1024,512)
    self.fc3 = nn.Linear(512,10)
    self.relu = nn.ReLU()
#ReLU   学習するパラメータがないので使いまわしできる
def forward(self,x):
  x = x.view(-1,img_size*img_size)
#画像を１次元に変換
  x = self.relu(self.fc1(x))
  x = self.relu(self.fc2(x))
  x = self.fc3(x)
  return x

net = Net()
net.cuda() #ランタイムでGPUを指定しているため、GPUを使用できる。
print(net)

Net(
  (fcl): Linear(in_features=784, out_features=1024, bias=True)
  (fc2): Linear(in_features=1024, out_features=512, bias=True)
  (fc3): Linear(in_features=512, out_features=10, bias=True)
  (relu): ReLU()
)


In [ ]:
from torch import optim

#交差エントロピー誤差関数
loss_fnc = nn.CrossEntropyLoss()

#SGD
optimizer = optim.SGD(net.parameters(), lr=0.01)

#損失を記録するリストの指定
record_loss_train = []
record_loss_test = []

#学習
for i in range(10):    #10エポック学習
    net.train()    #訓練モード
    loss_train = 0
    for j, (x, t) in enumerate(train_loader):    #ミニバッチ(x, t)を取り出す
      x, t = x.cuda(), t.cuda()    #GPU対応
      y = net(x)
